# Dyna-Q & SARSA vs Classical Pathfinding Algorithms

This notebook compares **Dyna-Q** and **SARSA** reinforcement learning agents with classical optimal pathfinding algorithms (A*, Dijkstra, BFS, Flood Fill) on Micromouse maze navigation.

## Evaluation Framework
- **ML Agents**: Dyna-Q (model-based with planning), SARSA (model-free on-policy)
- **Classical**: A*, Dijkstra, BFS, Flood Fill (optimal solutions)
- **Metrics**: Path length, computation time, success rate

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['figure.dpi'] = 100

from mazemind.envs.maze_parser import parse_maze_file, list_maze_files, load_random_maze
from mazemind.envs.micromouse_env import MicromouseEnv
from mazemind.agents.dyna_q import DynaQAgent
from mazemind.agents.sarsa import SarsaAgent
from mazemind.agents.classical import ClassicalSolver
from mazemind.training.orchestrator import train_with_snapshots, extract_optimal_path
from mazemind.visualization.maze_renderer import render_maze, render_maze_comparison

print('All imports successful.')

## 1. Load Maze

In [ ]:
maze_dir = os.path.join('data', 'mazes', 'classic')

files = list_maze_files(maze_dir)
if len(files) < 3:
    from mazemind.envs.maze_parser import download_mazes
    download_mazes(maze_dir)
    files = list_maze_files(maze_dir)

maze = load_random_maze(maze_dir)
print(f'Maze: {maze.name} ({maze.size}x{maze.size})')
print(f'Start: {maze.start} -> Goals: {maze.goals}')

fig, ax = render_maze(maze, title=f'Maze: {maze.name}')
plt.show()

## 2. Solve with Classical Algorithms

In [ ]:
solver = ClassicalSolver(maze)
classical_results = solver.solve_all()

print('Classical Algorithm Results')
print('=' * 50)
for result in classical_results:
    print(f"{result.algorithm:<12} | Path: {len(result.path):>4} steps | Found: {result.found}")
print('=' * 50)

## 3. Train ML Agents

In [ ]:
alpha = 0.1
gamma = 0.99
epsilon_start = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01
n_planning_steps = 10
n_episodes = 500
max_steps = 1000
seed = 42

print('Training Dyna-Q...')
dq_agent = DynaQAgent(
    n_planning_steps=n_planning_steps,
    epsilon=epsilon_start, epsilon_decay=epsilon_decay
)
dq_env = MicromouseEnv(maze)
dq_metrics, dq_snapshots, dq_traj, dq_exp = train_with_snapshots(
    dq_agent, dq_env, n_episodes=n_episodes, max_steps=max_steps,
    alpha=alpha, gamma=gamma, seed=seed,
    agent_name='Dyna-Q', maze_name=maze.name
)

print('Training SARSA...')
ss_agent = SarsaAgent(
    epsilon=epsilon_start, epsilon_decay=epsilon_decay
)
ss_env = MicromouseEnv(maze)
ss_metrics, ss_snapshots, ss_traj, ss_exp = train_with_snapshots(
    ss_agent, ss_env, n_episodes=n_episodes, max_steps=max_steps,
    alpha=alpha, gamma=gamma, seed=seed,
    agent_name='SARSA', maze_name=maze.name
)

dq_summary = dq_metrics.summary()
ss_summary = ss_metrics.summary()
print(f"\nDyna-Q: {dq_summary['success_rate']:.1%} success, {len(extract_optimal_path(dq_agent, MicromouseEnv(maze)))} steps")
print(f"SARSA: {ss_summary['success_rate']:.1%} success, {len(extract_optimal_path(ss_agent, MicromouseEnv(maze)))} steps")

## 4. Get Paths from ML Agents

In [ ]:
dq_path = extract_optimal_path(dq_agent, MicromouseEnv(maze))
ss_path = extract_optimal_path(ss_agent, MicromouseEnv(maze))

astar_path = next(r.path for r in classical_results if r.algorithm == 'A*')
dijkstra_path = next(r.path for r in classical_results if r.algorithm == 'Dijkstra')
bfs_path = next(r.path for r in classical_results if r.algorithm == 'BFS')
flood_path = next(r.path for r in classical_results if r.algorithm == 'Flood Fill')

all_paths = {
    'A*': astar_path,
    'Dijkstra': dijkstra_path,
    'BFS': bfs_path,
    'Flood Fill': flood_path,
    'Dyna-Q': dq_path,
    'SARSA': ss_path,
}

print('Path Lengths Comparison')
print('=' * 40)
for name, path in all_paths.items():
    print(f"{name:<12} {len(path):>4} steps")
print('=' * 40)

## 5. Visualize All Paths

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

titles = ['A*', 'Dijkstra', 'BFS', 'Flood Fill', 'Dyna-Q', 'SARSA']
paths = [astar_path, dijkstra_path, bfs_path, flood_path, dq_path, ss_path]

for i, (ax, title, path) in enumerate(zip(axes, titles, paths)):
    render_maze(maze, ax=ax, title=f'{title} ({len(path)} steps)', path=path)

plt.suptitle(f'Path Comparison: {maze.name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Success Rate Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

algorithms = ['A*', 'Dijkstra', 'BFS', 'Flood Fill', 'Dyna-Q', 'SARSA']
success_rates = [1.0, 1.0, 1.0, 1.0, dq_summary['success_rate'], ss_summary['success_rate']]

colors = ['#2ecc71'] * 4 + ['#3498db', '#e74c3c']

bars = ax.bar(algorithms, success_rates, color=colors, alpha=0.8)
ax.set_ylabel('Success Rate')
ax.set_title('Success Rate Comparison')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

for bar, rate in zip(bars, success_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
           f'{rate:.0%}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 7. Path Length Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

path_lengths = [len(p) for p in all_paths.values()]

bars = ax.bar(algorithms, path_lengths, color=colors, alpha=0.8)
ax.set_ylabel('Path Length (steps)')
ax.set_title('Path Length Comparison')
ax.grid(True, alpha=0.3, axis='y')

for bar, length in zip(bars, path_lengths):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
           str(length), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

optimal = min(path_lengths[:4])
print(f"\nOptimal path length (classical): {optimal} steps")
print(f"Dyna-Q path length: {len(dq_path)} steps ({len(dq_path)/optimal:.1%} of optimal)")
print(f"SARSA path length: {len(ss_path)} steps ({len(ss_path)/optimal:.1%} of optimal)")

## 8. Learning Curves with Optimal Markers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

window = 50
episodes = np.arange(n_episodes)

# Reward curves
axes[0].plot(episodes, dq_metrics.rewards, alpha=0.15, color='#3498db', linewidth=0.5)
smoothed_dq = dq_metrics.avg_reward(window)
axes[0].plot(episodes[window-1:], smoothed_dq, color='#3498db', linewidth=2, label='Dyna-Q')

axes[0].plot(episodes, ss_metrics.rewards, alpha=0.15, color='#e74c3c', linewidth=0.5)
smoothed_ss = ss_metrics.avg_reward(window)
axes[0].plot(episodes[window-1:], smoothed_ss, color='#e74c3c', linewidth=2, label='SARSA')

axes[0].axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Optimal (classical)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Cumulative Reward')
axes[0].set_title('Learning Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Success rate
sr_dq = dq_metrics.success_rate(window)
sr_ss = ss_metrics.success_rate(window)
axes[1].plot(episodes[window-1:], sr_dq * 100, color='#3498db', linewidth=2, label='Dyna-Q')
axes[1].plot(episodes[window-1:], sr_ss * 100, color='#e74c3c', linewidth=2, label='SARSA')
axes[1].axhline(y=100, color='green', linestyle='--', alpha=0.5, label='100% (optimal)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Success Rate (%)')
axes[1].set_title('Success Rate')
axes[1].set_ylim(-5, 105)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('ML Agent Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Summary Table

In [ ]:
print('\n' + '=' * 80)
print(f"{'Algorithm':<12} | {'Path Length':>12} | {'Success Rate':>12} | {'Type':<12}")
print('-' * 80)
for alg in algorithms:
    path_len = len(all_paths[alg])
    if alg in ['A*', 'Dijkstra', 'BFS', 'Flood Fill']:
        success = '100%'
        alg_type = 'Classical'
    else:
        success = f"{dq_summary['success_rate']:.0%}" if alg == 'Dyna-Q' else f"{ss_summary['success_rate']:.0%}"
        alg_type = 'ML (RL)'
    print(f"{alg:<12} | {path_len:>12} | {success:>12} | {alg_type:<12}")
print('=' * 80)

print("\nKey Insights:")
print("- Classical algorithms find the optimal path 100% of the time")
print("- Dyna-Q and SARSA learn to find near-optimal paths through exploration")
print("- RL agents adapt to changing environments; classical are static solvers")